In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

train_df = pd.read_csv("train_split.csv")
test_df = pd.read_csv("test_split.csv")

X_text_train = train_df["clean_text"]
X_text_test = test_df["clean_text"]

y_train = train_df["label"]
y_test = test_df["label"]

vectorizer_40000 = CountVectorizer(max_features=40000)

X_train_40000 = vectorizer_40000.fit_transform(X_text_train)
X_test_40000 = vectorizer_40000.transform(X_text_test)

print(X_train_40000.shape)
print(X_test_40000.shape)

(6400, 40000)
(1600, 40000)


In [2]:
from sklearn.svm import LinearSVC

model = LinearSVC()

model.fit(X_train_40000, y_train)

y_pred = model.predict(X_test_40000)

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [3]:
from sklearn.svm import LinearSVC

model = LinearSVC()

model.fit(X_train_40000, y_train)

y_pred = model.predict(X_test_40000)

print(y_pred[:20])

['Not Relevant' 'Not Relevant' 'Not Relevant' 'Not Relevant'
 'Not Relevant' 'Relevant' 'Relevant' 'Not Relevant' 'Not Relevant'
 'Not Relevant' 'Not Relevant' 'Not Relevant' 'Not Relevant'
 'Not Relevant' 'Not Relevant' 'Relevant' 'Not Relevant' 'Not Relevant'
 'Not Relevant' 'Not Relevant']


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label="Relevant",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="Relevant",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="Relevant",
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Not Relevant", "Relevant"]
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.755625
Precision: 0.31864406779661014
Recall   : 0.33098591549295775
F1-score : 0.32469775474956825

Confusion Matrix:
[[1115  201]
 [ 190   94]]


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_svc(model, X_train, X_test, y_train, y_test):

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        pos_label="Relevant",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        pos_label="Relevant",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        pos_label="Relevant",
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=["Not Relevant", "Relevant"]
    )

    return accuracy, precision, recall, f1, cm

In [6]:
model = LinearSVC()

accuracy, precision, recall, f1, cm = evaluate_svc(
    model,
    X_train_40000,
    X_test_40000,
    y_train,
    y_test
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.75625
Precision: 0.3197278911564626
Recall   : 0.33098591549295775
F1-score : 0.32525951557093424

Confusion Matrix:
[[1116  200]
 [ 190   94]]


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [7]:
vectorizer_5000 = CountVectorizer(max_features=5000)

X_train_5000 = vectorizer_5000.fit_transform(X_text_train)
X_test_5000 = vectorizer_5000.transform(X_text_test)

print(X_train_5000.shape)
print(X_test_5000.shape)

(6400, 5000)
(1600, 5000)


In [8]:
vectorizer_1000 = CountVectorizer(max_features=1000)

X_train_1000 = vectorizer_1000.fit_transform(X_text_train)
X_test_1000 = vectorizer_1000.transform(X_text_test)

print(X_train_1000.shape)
print(X_test_1000.shape)

(6400, 1000)
(1600, 1000)


In [9]:
feature_sets = {
    40000: (X_train_40000, X_test_40000),
    5000: (X_train_5000, X_test_5000),
    1000: (X_train_1000, X_test_1000)
}

In [11]:
results = []

for n_features, (X_train, X_test) in feature_sets.items():

    # -------------------------
    # Normal LinearSVC
    # -------------------------
    model = LinearSVC(max_iter=5000)

    accuracy, precision, recall, f1, cm = evaluate_svc(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    results.append({
        "Model": "LinearSVC",
        "Features": n_features,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Confusion Matrix": cm
    })


    # -------------------------
    # Balanced LinearSVC
    # -------------------------
    model = LinearSVC(class_weight="balanced", max_iter=5000)

    accuracy, precision, recall, f1, cm = evaluate_svc(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    results.append({
        "Model": "LinearSVC (Balanced)",
        "Features": n_features,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Confusion Matrix": cm
    })

In [12]:
for result in results:
    print("\n", result["Model"])
    print("Features :", result["Features"])
    print("Accuracy :", result["Accuracy"])
    print("Precision:", result["Precision"])
    print("Recall   :", result["Recall"])
    print("F1       :", result["F1"])
    print("Confusion Matrix:")
    print(result["Confusion Matrix"])


 LinearSVC
Features : 40000
Accuracy : 0.754375
Precision: 0.3152542372881356
Recall   : 0.3274647887323944
F1       : 0.32124352331606215
Confusion Matrix:
[[1114  202]
 [ 191   93]]

 LinearSVC (Balanced)
Features : 40000
Accuracy : 0.7525
Precision: 0.31333333333333335
Recall   : 0.33098591549295775
F1       : 0.3219178082191781
Confusion Matrix:
[[1110  206]
 [ 190   94]]

 LinearSVC
Features : 5000
Accuracy : 0.73625
Precision: 0.3017241379310345
Recall   : 0.36971830985915494
F1       : 0.3322784810126582
Confusion Matrix:
[[1073  243]
 [ 179  105]]

 LinearSVC (Balanced)
Features : 5000
Accuracy : 0.7325
Precision: 0.2988826815642458
Recall   : 0.3767605633802817
F1       : 0.3333333333333333
Confusion Matrix:
[[1065  251]
 [ 177  107]]

 LinearSVC
Features : 1000
Accuracy : 0.791875
Precision: 0.3515151515151515
Recall   : 0.20422535211267606
F1       : 0.2583518930957684
Confusion Matrix:
[[1209  107]
 [ 226   58]]

 LinearSVC (Balanced)
Features : 1000
Accuracy : 0.6775
Prec

In [13]:
import pandas as pd

results_table = []

for result in results:
    cm = result["Confusion Matrix"]

    results_table.append({
        "Model": result["Model"],
        "Features": result["Features"],
        "Accuracy": result["Accuracy"],
        "Precision": result["Precision"],
        "Recall": result["Recall"],
        "F1": result["F1"],
        "TN": cm[0][0],
        "FP": cm[0][1],
        "FN": cm[1][0],
        "TP": cm[1][1]
    })

results_df = pd.DataFrame(results_table)

results_df

,Model,Features,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,LinearSVC,40000,0.754375,0.315254,0.327465,0.321244,1114,202,191,93
1,LinearSVC (Balanced),40000,0.752500,0.313333,0.330986,0.321918,1110,206,190,94
2,LinearSVC,5000,0.736250,0.301724,0.369718,0.332278,1073,243,179,105
3,LinearSVC (Balanced),5000,0.732500,0.298883,0.376761,0.333333,1065,251,177,107
4,LinearSVC,1000,0.791875,0.351515,0.204225,0.258352,1209,107,226,58
5,LinearSVC (Balanced),1000,0.677500,0.281132,0.524648,0.366093,935,381,135,149


In [14]:
results_df.to_csv(
    "member5_linearsvc_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!
